### COMMON INSTRUCTIONS
Make sure to follow these instructions in order to ensure your submission is gradeable. If the automatic grading fails (check via Validate), no points are assigned!
1. Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel $\rightarrow$ Restart) and then **run all cells** (in the menubar, select Cell $\rightarrow$ Run All).
2. Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE".
3. If you work in a group, please always work and submit in a shared server under the name of your group.
   If you work alone, then always work and submit on the server called "My Server".
4. Using Large Language Models and other AI tools is discouraged. Do not paste external code, as it can be tracked!
5. Do not upload notebooks. Always work in notebooks on the server.
6. Do not copy cells inside a notebook. Add a new cell and copy code from another cell if needed.

In [ ]:
!tar czf myfiles.tar.gz ./*

---

# eXplainable AI, Assignment 6
# Ulm University - Institute of Artificial Intelligence
# Sommersemester 2026 - Due: May 27, 2026, 2pm - Moodle


# Data preparation

Please run the code below to load and prepare the dataset.

In [ ]:
! pip install scikit-learn pandas matplotlib

In [ ]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

dataset = pd.read_csv("housing_prices.csv", index_col=0).sample(n=200, random_state=1)

FEATURE_COLUMNS = [
    "sqft_living",
    "sqft_lot",
    "bedrooms",
    "bathrooms",
    "floors",
    "yr_built",
    "grade",
]
TARGET_COLUMN = ["price"]

X_train, X_test, y_train, y_test = train_test_split(
    dataset[FEATURE_COLUMNS], dataset[TARGET_COLUMN], train_size=0.66, random_state=1337
)
index_train, index_test = X_train.index, X_test.index
scaler = StandardScaler().fit(X_train)
X_train, X_test = pd.DataFrame(scaler.transform(X_train), columns=FEATURE_COLUMNS, index=index_train), pd.DataFrame(scaler.transform(X_test), columns=FEATURE_COLUMNS, index=index_test)

regressor = RandomForestRegressor(
    max_depth=4,
    n_estimators=50,
    random_state=1337,
)
regressor.fit(X_train.values, y_train.values.ravel())


train_r2 = r2_score(y_train, regressor.predict(X_train.values))
test_r2 = r2_score(y_test, regressor.predict(X_test.values))

print(f"R2 metric on train set: {train_r2:.4f}, on test set: {test_r2:.4f}")

# 1. Prototypes & Critics

In this exercise we want to take a deeper look at selecting prototypes and criticisms. We will use the simple MMD-critic method for that (for details see the [original paper](https://proceedings.neurips.cc/paper/2016/hash/5680522b8e2bb01943234bce7bf84534-Abstract.html) or [Molnar's summary](https://christophm.github.io/interpretable-ml-book/proto.html#theory)).

## 1.1 MMD-critic (3 pts)
Implement a greedy search for prototypes and find five prototypes with it. You may use the implementation of MMD2 provided below.


In [ ]:
# Helper methods
from sklearn.metrics.pairwise import rbf_kernel


def MMD2(x: np.ndarray, y: np.ndarray):
    """Calculate the squared MMD metric as distance between the distributions of datasets x and y."""
    x1x1 = rbf_kernel(x, x).sum(axis=-1)
    x1x2 = rbf_kernel(x, y).sum(axis=-1)
    x2x2 = rbf_kernel(y, y).sum(axis=-1)
    mmd = (
        ((1 / x.shape[0] ** 2) * x1x1.sum())
        - ((2 / (x.shape[0] * y.shape[0])) * x1x2.sum())
        + ((1 / y.shape[0] ** 2) * x2x2.sum())
    )
    return mmd

In [ ]:
def find_prototypes(X: pd.DataFrame, num_prototypes: int) -> tuple[list[pd.Series], float]:
    """Iteratively collect the num_prototype prototypes from data X that minimize MMD2 between prototypes and data distribution.
    Returns the list of prototypes together with their MMD2 value against the data distribution.
    """
    candidates = []
    current_mmd = np.inf  # just some very large value (no prototypes to compare with)
    
    # YOUR SOLUTION HERE

    return candidates, float(current_mmd)


prototypes, mmd2 = find_prototypes(X_train, num_prototypes=5)
prototypes = pd.DataFrame(prototypes)
prototypes

In [ ]:
# (hidden tests)
print('Success!')

## 1.2 Controlling MMD$^2$ (1+0.5 pts)
Explore the influence of the number of prototypes on the faithfulness of the explanation: Plot the number of prototypes against their respective MMD2 distance metric (the lower the more faithful) for the first 100 samples of the training set. Which number of prototypes would you choose?

*Hint:* There is no need to go beyond 15.

In [ ]:
X_sub = X_train.iloc[:100]
# YOUR SOLUTION HERE

Recommended choice for the number of prototypes:
YOUR ANSWER HERE

## 1.2 Criticisms (2 pts)
Implement the witness scores. Use it to select a number of five critics.
You can use the `rbf_kernel(points1, points2)` function as a kernel in the witness score.

In [ ]:
def witness_score(X: pd.DataFrame, prototypes: pd.DataFrame, query: pd.Series) -> float:
    """Calculate the witness function for query point against the distributions of data points X and prototypes."""
    witness_score = 0
    # YOUR SOLUTION HERE
    return witness_score


# Collect the scores and select criticisms from the training data accordingly
scores = [
    witness_score(X_train, prototypes, point) for _, point in X_train.iterrows()
]
critic_indices = np.argsort(np.abs(scores))[:5]
critic_scores = np.asarray(scores)[critic_indices]
criticisms = X_train.iloc[critic_indices, :].copy()
pd.DataFrame(criticisms.to_dict() | {"witness_score": pd.Series(critic_scores, index = criticisms.index)})

In [ ]:
# (hidden tests)
print('Success!')

## 1.3 Properties of Criticisms (1 + 0.5 pts)
When is a witness score negative? When positive?
What does a sample have to fulfill to become a negative-scored criticism?

Bonus: What makes each of the criticisms a criticism? Can you spot values that are particularly different?

YOUR ANSWER HERE

In [ ]:
# YOUR SOLUTION HERE

# 2. Counterfactual example

In the lecture we have foremostly had a look at techniques for example-based counterfactual generation. Here we will pursue a simpler selection scheme: The nearest unlike neighbor.

## 2.1 Simple Counterfactuals (3 pts)
We now apply a simple adaption of the nearest unlike neighbor idea to our dataset's regression task.
Implement a function that will pick for a given query sample the most proximate sample in the dataset which has an output value of at least `min_eps=5000` more than the query.
Use Manhattan distance for proximity.

Determine a counterfactual for the training data point at index #1.

In [ ]:
from sklearn.metrics.pairwise import manhattan_distances

def nearest_unlike_neighbor(query: pd.Series, y_pred_query: float, X: pd.DataFrame, black_box, min_eps: float = 5000.):
    nun = None
    # YOUR SOLUTION HERE
    return nun

query = X_train.iloc[1]
y_pred_query = regressor.predict([query.values])
nun = nearest_unlike_neighbor(query, y_pred_query, X_train, regressor, min_eps = 5000)
y_pred_nun = regressor.predict([nun.values])
nun - query
pd.DataFrame(query.to_dict() | {'prediction': y_pred_query}, index = ["original"])
pd.DataFrame(nun.to_dict() | {'prediction': y_pred_nun}, index = ["counterfact"])

In [ ]:
# (hidden tests)
print('Success!')

## 2.2 Limitations (1pt)
Name a limitation of above technique for determining counterfactuals.

YOUR ANSWER HERE